# 12 — Comment Extraction Training Data

Extracts every `(* ... *)` comment from `.aro` files in `Examples/` and
`../ARO-Application/` and pairs each comment with the code it documents.

Two complementary datasets:

| Direction | Instruction template | Target |
|-----------|---------------------|--------|
| code -> comment | `Explain / Describe / What does / Summarise <code>` | comment text |
| comment -> code | model-reworded comment as imperative instruction | code |

The local model generates **4 paraphrase variations** of every instruction.
Each raw comment-code pair produces 13 training samples (4 fixed + 4 paraphrase + 1 base + 4 paraphrase).

## Setup

In [ ]:
import sys, importlib, json, re, random
from pathlib import Path

_cfg_dir = Path('.').resolve()
if str(_cfg_dir) not in sys.path:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *

ARO_APPLICATION_ROOT = Path('/Users/kris/Projects/ARO-Application')
OUT_FILE = SCRIPT_DIR / 'comment_pairs.jsonl'

SYSTEM = (
    "You are an expert ARO (Action Result Object) programmer. "
    "ARO is a DSL where every statement is: Verb the <Result> preposition [the] <Object>. "
    "Feature sets follow (Name: Business Activity) { statements }. "
    "Variables are immutable. String concatenation uses ++."
)

print('Setup complete.')
print(f'  Examples root        : {EXAMPLES_DIR}')
print(f'  ARO-Application root : {ARO_APPLICATION_ROOT}')
print(f'  Output file          : {OUT_FILE}')

## Load model

In [ ]:
model, tokenizer, _load_fn, mlx_generate, make_sampler = load_model(with_adapter=False)
print('Model ready.')

## Chat helper

In [ ]:
def chat(messages: list[dict], max_tokens: int = 200, temperature: float = 0.7) -> str:
    sampler = make_sampler(temp=temperature)
    prompt  = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return mlx_generate(
        model, tokenizer, prompt=prompt,
        max_tokens=max_tokens, sampler=sampler, verbose=False,
    ).strip()

## Comment extraction

In [ ]:
# Matches (* ... *) — single-line and multi-line
_COMMENT_RE = re.compile(r'\(\*(.*?)\*\)', re.DOTALL)

MIN_CODE_LINES = 1
MAX_CODE_LINES = 8


def _clean_comment(raw: str) -> str:
    lines = []
    for ln in raw.splitlines():
        ln = ln.strip().lstrip('*').strip()
        if ln:
            lines.append(ln)
    return ' '.join(lines).strip()


def _is_file_header(text: str, comment_idx: int) -> bool:
    """Skip the very first comment when it looks like a file-level header."""
    return comment_idx == 0 and len(text) > 200


def extract_comment_code_pairs(path: Path) -> list[dict]:
    """
    Return [{comment, code, source}] for every inline comment
    followed by at least MIN_CODE_LINES of ARO code.
    """
    try:
        src = path.read_text(encoding='utf-8', errors='ignore')
    except Exception:
        return []

    pairs = []
    comment_idx = 0

    for m in _COMMENT_RE.finditer(src):
        raw_comment  = m.group(1)
        comment_text = _clean_comment(raw_comment)

        if not comment_text or len(comment_text) < 15:
            continue
        if _is_file_header(comment_text, comment_idx):
            comment_idx += 1
            continue
        comment_idx += 1

        after      = src[m.end():]
        code_lines = []
        for ln in after.splitlines():
            stripped = ln.strip()
            if not stripped:
                continue
            if stripped.startswith('(*'):
                break
            if stripped.startswith('(') and ':' in stripped and stripped.endswith('{'):
                break
            if stripped == '}':
                break
            code_lines.append(stripped)
            if len(code_lines) >= MAX_CODE_LINES:
                break

        if len(code_lines) < MIN_CODE_LINES:
            continue

        pairs.append({
            'comment': comment_text,
            'code':    '\n'.join(code_lines),
            'source':  str(path),
        })

    return pairs


def mine_all_files() -> list[dict]:
    all_pairs = []
    for label, root in [('Examples', EXAMPLES_DIR), ('ARO-Application', ARO_APPLICATION_ROOT)]:
        files       = sorted(root.rglob('*.aro'))
        cnt_files   = 0
        cnt_pairs   = 0
        for f in files:
            fp = extract_comment_code_pairs(f)
            if fp:
                cnt_files += 1
                cnt_pairs += len(fp)
                all_pairs.extend(fp)
        print(f'  {label:<20}  {cnt_files:>3} files with comments  ->  {cnt_pairs:>4} raw pairs')
    print(f'\nTotal raw comment-code pairs: {len(all_pairs)}')
    return all_pairs


raw_pairs = mine_all_files()

## Inspect raw pairs

In [ ]:
sample = random.sample(raw_pairs, min(5, len(raw_pairs)))
for p in sample:
    print(f"SOURCE : {Path(p['source']).name}")
    print(f"COMMENT: {p['comment'][:120]}")
    print('CODE   :')
    print(p['code'])
    print()

## Instruction paraphrase helpers

**Code -> Comment**: 4 fixed templates (Explain / Describe / What does / Summarise)
plus 4 model paraphrases of the Explain form.

**Comment -> Code**: model first rewrites the comment as an imperative instruction,
then generates 4 paraphrase variants of that instruction.

In [ ]:
_PARA_SYSTEM = (
    "You are a dataset augmentation assistant. "
    "Given an instruction about ARO code, rewrite it in a different but semantically "
    "equivalent way. Return ONLY the rewritten instruction — no explanation, no quotes."
)

_CODE2COMMENT_TEMPLATES = [
    "Explain what this ARO code does:\n\n```aro\n{code}\n```",
    "Describe the purpose of the following ARO code:\n\n```aro\n{code}\n```",
    "What does this ARO snippet do?\n\n```aro\n{code}\n```",
    "Summarise this ARO code in plain language:\n\n```aro\n{code}\n```",
]


def _code2comment_fixed(code: str) -> list[str]:
    """4 fixed template instructions for code -> comment."""
    return [t.format(code=code) for t in _CODE2COMMENT_TEMPLATES]


def _paraphrase(instruction: str, n: int = 4) -> list[str]:
    """Return n paraphrase variations of instruction."""
    results = []
    for _ in range(n):
        out = chat([
            {'role': 'system', 'content': _PARA_SYSTEM},
            {'role': 'user',   'content':
                f'Rewrite this instruction as a different imperative sentence '
                f'describing the same ARO code. Instruction: {instruction}'},
        ], max_tokens=120, temperature=0.85).strip().strip('"').strip("'")
        if out and out not in results:
            results.append(out)
    while len(results) < n:
        results.append(instruction)
    return results[:n]


def _comment_to_instruction(comment: str) -> str:
    """Reword a comment into an imperative code-generation instruction."""
    return chat([
        {'role': 'system', 'content':
            "You are a technical writing assistant. "
            "Turn the following ARO code comment into a concise imperative instruction "
            "(1-2 sentences) that tells a developer what ARO code to write. "
            "Start with a verb. Return ONLY the instruction."},
        {'role': 'user', 'content': comment},
    ], max_tokens=100, temperature=0.6)


print('Helpers ready.')

## Build training pairs

Per raw comment-code pair:

- **4** code->comment fixed templates
- **4** code->comment model paraphrases
- **1** comment->code base instruction  
- **4** comment->code paraphrase variants

= **13 training samples** per raw pair.

In [ ]:
# Resume support
done_keys: set[str] = set()
all_pairs: list[dict] = []

if OUT_FILE.exists():
    for line in OUT_FILE.read_text().splitlines():
        if line.strip():
            try:
                p = json.loads(line)
                done_keys.add(p.get('source_key', ''))
                all_pairs.append(p)
            except Exception:
                pass
    print(f'Resumed: {len(all_pairs)} pairs, {len(done_keys)} source keys done.')
else:
    print('Starting fresh.')


total = len(raw_pairs)
for i, rp in enumerate(raw_pairs):
    skey = f"{rp['source']}::{i}"
    if skey in done_keys:
        continue

    comment = rp['comment']
    aro_code = rp['code']
    source   = rp['source']

    def _pair(instr, reply, itype):
        return {
            'messages': [
                {'role': 'system',    'content': SYSTEM},
                {'role': 'user',      'content': instr},
                {'role': 'assistant', 'content': reply},
            ],
            'task_type':        'code_explanation' if itype.startswith('code2comment') else 'code_generation',
            'instruction_type': itype,
            'source_key':       skey,
            'source':           source,
        }

    # A. Code -> Comment: 4 fixed templates
    for instr in _code2comment_fixed(aro_code):
        all_pairs.append(_pair(instr, comment, 'code2comment_template'))

    # B. Code -> Comment: 4 model paraphrases
    base_explain = f'Explain what this ARO code does:\n\n```aro\n{aro_code}\n```'
    for variant in _paraphrase(base_explain, n=4):
        all_pairs.append(_pair(variant, comment, 'code2comment_paraphrase'))

    # C. Comment -> Code: base instruction + 4 paraphrases
    base_c2code = _comment_to_instruction(comment)
    all_pairs.append(_pair(base_c2code, f'```aro\n{aro_code}\n```', 'comment2code_base'))
    for variant in _paraphrase(base_c2code, n=4):
        all_pairs.append(_pair(variant, f'```aro\n{aro_code}\n```', 'comment2code_paraphrase'))

    done_keys.add(skey)

    # Incremental save every 10 source pairs
    if (i + 1) % 10 == 0 or i == total - 1:
        with OUT_FILE.open('w') as f:
            for p in all_pairs:
                f.write(json.dumps(p) + '\n')
        print(f'  [{i+1}/{total}] {len(all_pairs)} pairs saved', flush=True)

print(f'\nDone. {len(all_pairs)} total training pairs -> {OUT_FILE}')

## Inspect output

In [ ]:
from collections import Counter

_saved: list[dict] = []
for line in OUT_FILE.read_text().splitlines():
    if line.strip():
        try: _saved.append(json.loads(line))
        except Exception: pass

by_type = Counter(p['instruction_type'] for p in _saved)
print(f'Total pairs: {len(_saved)}')
for k, v in sorted(by_type.items()):
    print(f'  {k:<35} {v:>5}')

print()
for p in random.sample(_saved, min(3, len(_saved))):
    print(f"TYPE : {p['instruction_type']}")
    print(f"USER : {p['messages'][1]['content'][:140]}")
    print(f"ASST : {p['messages'][2]['content'][:140]}")
    print()

## Append to knowledge_pairs.jsonl

In [ ]:
PAIRS_FILE.parent.mkdir(parents=True, exist_ok=True)

existing = len([l for l in PAIRS_FILE.read_text().splitlines() if l.strip()]) if PAIRS_FILE.exists() else 0
new_lines = [l for l in OUT_FILE.read_text().splitlines() if l.strip()]

with PAIRS_FILE.open('a') as out:
    for line in new_lines:
        out.write(line + '\n')

print(f'Appended {len(new_lines)} pairs to {PAIRS_FILE}')
print(f'  Before: {existing}  After: {existing + len(new_lines)}')

## Summary

In [ ]:
print('=' * 60)
print('Comment Extraction Training Data — Summary')
print('=' * 60)

_all: list[dict] = []
for line in OUT_FILE.read_text().splitlines():
    if line.strip():
        try: _all.append(json.loads(line))
        except Exception: pass

c2c    = [p for p in _all if p['task_type'] == 'code_explanation']
c2code = [p for p in _all if p['task_type'] == 'code_generation']

print(f'\nRaw comment-code pairs mined : {len(done_keys)}')
print(f'\nCode -> Comment pairs        : {len(c2c)}')
print(f'  template    : {sum(1 for p in c2c    if p["instruction_type"] == "code2comment_template")}')
print(f'  paraphrase  : {sum(1 for p in c2c    if p["instruction_type"] == "code2comment_paraphrase")}')
print(f'\nComment -> Code pairs        : {len(c2code)}')
print(f'  base        : {sum(1 for p in c2code if p["instruction_type"] == "comment2code_base")}')
print(f'  paraphrase  : {sum(1 for p in c2code if p["instruction_type"] == "comment2code_paraphrase")}')
print(f'\nTotal                        : {len(_all)}')
print(f'Output file                  : {OUT_FILE}')

In [ ]:
import matplotlib.pyplot as plt

labels = ['code->comment\ntemplate', 'code->comment\nparaphrase',
          'comment->code\nbase',     'comment->code\nparaphrase']
values = [
    sum(1 for p in _all if p['instruction_type'] == 'code2comment_template'),
    sum(1 for p in _all if p['instruction_type'] == 'code2comment_paraphrase'),
    sum(1 for p in _all if p['instruction_type'] == 'comment2code_base'),
    sum(1 for p in _all if p['instruction_type'] == 'comment2code_paraphrase'),
]
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=0.8)
ax.bar_label(bars, padding=4, fontsize=10)
ax.set_title('12 — Comment Extraction: pairs by type', fontsize=13, pad=12)
ax.set_ylabel('# pairs')
ax.set_ylim(0, max(values) * 1.25 if values else 10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()